In [ ]:
# When running in Google Colab, this cell installs dependencies, clones the
# paper repository, and downloads the required data.  The setup takes about a
# minute; downloading the fit pkl files (main fit + 120 bootstraps) may add
# several more minutes depending on file sizes and network speed.
# No-op when running locally.
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    # 1. Clone wppmpy (provides download_data.py and correct path structure)
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/BrainardLab/wppmpy.git"],
        check=True,
    )

    # 2. Clone the paper repo and add to sys.path (for analysis.* imports)
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/fh862/ellipsoids_eLife2025.git",
        ],
        check=True,
    )
    sys.path.insert(0, "/content/ellipsoids_eLife2025")

    # 3. Install paper repo dependencies (jax, scipy, dill, shapely, ...)
    subprocess.run(
        [
            "pip",
            "install",
            "-q",
            "git+https://github.com/fh862/ellipsoids_eLife2025.git",
        ],
        check=True,
    )

    # 4. Download CSV tables, calibration matrices, and fit pkl files
    subprocess.run(
        [
            "python",
            "/content/wppmpy/src/hong_etal_2025/download_data.py",
            "--data-dir",
            "/content/wppmpy/data/hong_etal_2025",
            "--fits",
        ],
        check=True,
    )

    # 5. Set CWD so pathlib.Path("../...") resolves to the wppmpy repo root
    os.chdir("/content/wppmpy/src/hong_etal_2025")

# Threshold Ellipses from Wishart Process Fit Parameters

Reproduces **Figure 2C** of Hong et al. (2025) by loading the fitted WPPM object
from its pickle file rather than reading pre-tabulated CSV matrices.

The pipeline demonstrated here is:

1. **Load fit** — the fitted `wishart_model_pred` object contains the estimated
   Chebyshev basis weights `W_est` and model hyperparameters.
2. **Noise covariances** — `W_est` is evaluated at each grid point to produce
   the noise covariance field $\Sigma_\mathrm{noise}(x)$ via the Wishart process mapping.
3. **Threshold covariances** — converting noise covariances to threshold covariances
   (the 66.7 % correct discrimination contour) requires simulating the oddity task
   and is expensive; these have been pre-computed and stored in the pkl object.
4. **Bootstrap CI** — 120 bootstrap fit objects provide alternative threshold fields;
   the 95 % confidence band is the radial envelope across the top-ranked 114 fits.

Compare with `ellipses_from_tables.ipynb`, which reads $\Sigma_\mathrm{thres}$ directly
from CSV tables without touching the pkl.

## Subject mapping

| sub | initials |
|-----|----------|
| 1   | CH       |
| 2   | ME       |
| 4   | SG       |
| 6   | DK       |
| 7   | BH       |
| 8   | FM       |
| 10  | HG       |
| 11  | FW       |

## Before running

Download the required data files once (includes fit pkl files):

```bash
python src/hong_etal_2025/download_data.py --fits
```

## 1. Setup

In [ ]:
# ruff: noqa: E402
import glob as glob_mod
import importlib.util
import pathlib
import sys
import types
import warnings
from collections.abc import Iterator
from typing import Any

# ── Tkinter stubs ─────────────────────────────────────────────────────────────
# tkinter's __init__.py runs `import _tkinter` (a C extension).  If Tk is not
# installed we must stub both before anything else imports them, otherwise
# Python starts executing tkinter/__init__.py and hits the missing C extension.


class _TkStub(types.ModuleType):
    """Stub module that auto-creates sub-stubs for any attribute access."""

    TclError = type("TclError", (Exception,), {})

    def __init__(self, name: str) -> None:
        super().__init__(name)
        self.__path__: list[str] = []  # importlib needs iterable __path__

    def __call__(self, *a: object, **kw: object) -> "types.ModuleType":
        return self

    def __enter__(self) -> "types.ModuleType":
        return self

    def __exit__(self, *a: object) -> None:
        pass

    def __iter__(self) -> Iterator[Any]:
        return iter([])

    def __bool__(self) -> bool:
        return False

    def __getattr__(self, name: str) -> "types.ModuleType":
        full = f"{self.__name__}.{name}"
        sub = _TkStub(full)
        setattr(self, name, sub)
        sys.modules[full] = sub
        return sub


if "_tkinter" not in sys.modules:
    _tk_c = types.ModuleType("_tkinter")
    _tk_c.TclError = type("TclError", (Exception,), {})  # type: ignore[attr-defined]
    sys.modules["_tkinter"] = _tk_c
if "tkinter" not in sys.modules:
    sys.modules["tkinter"] = _TkStub("tkinter")

# ── Targeted stubs for packages absent in this environment ───────────────────
# The pkl was saved in an environment with torch and an older jaxlib that
# exposed jaxlib.xla_extension.  We stub only these specific prefixes so that
# installed packages (jax, scipy, …) are handled by the normal import machinery.

_STUB_PREFIXES = ("torch", "jaxlib.xla_extension")


class _TargetedStubFinder:
    """Meta-path finder that stubs only known-missing module prefixes."""

    def find_spec(
        self, fullname: str, path: object, target: object = None
    ) -> importlib.machinery.ModuleSpec | None:
        for pfx in _STUB_PREFIXES:
            if fullname == pfx or fullname.startswith(pfx + "."):
                return importlib.util.spec_from_loader(
                    fullname,
                    self,  # type: ignore[arg-type]
                )
        return None

    def create_module(self, spec: object) -> _TkStub:
        return _TkStub(spec.name)  # type: ignore[attr-defined]

    def exec_module(self, module: object) -> None:
        pass


sys.meta_path.append(_TargetedStubFinder())

import matplotlib

# In Colab there is no display server, so use the non-interactive Agg backend.
# Locally (Jupyter) we leave the backend alone so figures render inline.
if "google.colab" in sys.modules:
    matplotlib.use("Agg")

import dill
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
from scipy.linalg import sqrtm as scipy_sqrtm

jax.config.update("jax_enable_x64", True)  # required by the paper repo

from analysis.color_thres import color_thresholds

# ── Robust pkl loader ─────────────────────────────────────────────────────────
# The pkl references classes from packages present when it was saved but absent
# or renamed here (e.g. jax.errors.SimplifiedTraceback, removed in jax ≥ 0.5).
# _RobustUnpickler catches any find_class failure and returns a harmless stub
# class so the rest of the object graph can be restored.


class _RobustUnpickler(dill.Unpickler):  # type: ignore[misc]
    """dill Unpickler that returns a stub for any class not found at load time."""

    def find_class(self, module: str, name: str) -> Any:
        try:
            return super().find_class(module, name)
        except (AttributeError, ModuleNotFoundError, ImportError):
            return type(name, (), {"__module__": module})


# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = pathlib.Path("../..")
DATA_DIR = REPO_ROOT / "data" / "hong_etal_2025"

# Subject to plot
SUB_N = 1  # 1=CH, 2=ME, 4=SG, 6=DK, 7=BH, 8=FM, 10=HG, 11=FW
SUB_INITIALS = {
    1: "CH",
    2: "ME",
    4: "SG",
    6: "DK",
    7: "BH",
    8: "FM",
    10: "HG",
    11: "FW",
}
PLANE = "Isoluminant plane"

## 2. Load the Wishart process fit

The pkl file contains a `wishart_model_pred` object (`vars_dict["model_pred_Wishart"]`)
with the following key attributes:

- **`model`** — a `WishartProcessModel` instance storing the hyperparameters
  (Chebyshev degree, decay rate, variance scale, diagonal term) and the
  `compute_U` / `compute_Sigmas` methods.
- **`W_est`** — the MAP-estimated Chebyshev basis weights,
  shape `(degree, degree, num_dims_cov, num_dims_cov + extra_dims)`.
- **`Sigmas_noise_grid`** — noise covariance matrices on the 7×7 reference grid,
  shape `(7, 7, 2, 2)`.
- **`Sigmas_thres_grid`** — pre-computed threshold covariance matrices on the
  same grid, shape `(7, 7, 2, 2)`.

In [ ]:
PKL_DIR = (
    DATA_DIR
    / "Organized data and model predictions"
    / f"sub{SUB_N}"
    / "analyzed data files with class objects"
)

# Locate the main (non-bootstrap) pkl file
all_pkls = sorted(glob_mod.glob(str(PKL_DIR / "*.pkl")))
main_pkls = [p for p in all_pkls if "btst" not in pathlib.Path(p).name]
if not main_pkls:
    raise FileNotFoundError(
        f"No main pkl found in {PKL_DIR}.\n"
        "Run: python src/hong_etal_2025/download_data.py --fits"
    )

pkl_path = pathlib.Path(main_pkls[0])
print(f"Loading: {pkl_path.name}")

with open(pkl_path, "rb") as fh:
    vars_dict = _RobustUnpickler(fh).load()

model_pred = vars_dict["model_pred_Wishart"]
model = model_pred.model
W_est = model_pred.W_est

print("\nModel hyperparameters:")
print(f"  degree         = {model.degree}")
print(f"  num_dims       = {model.num_dims}")
print(f"  extra_dims     = {model.extra_dims}")
print(f"  variance_scale = {model.variance_scale}")
print(f"  decay_rate     = {model.decay_rate}")
print(f"  diag_term      = {model.diag_term}")
print(f"\nW_est shape:             {np.array(W_est).shape}")
print(f"Sigmas_noise_grid shape: {np.array(model_pred.Sigmas_noise_grid).shape}")

## 3. Stimulus grid and noise covariance field

The Wishart process model maps each point $x$ in the 2-D W-space to a positive
definite noise covariance matrix via:

$$U(x) = \sum_{ij} W_{ij}\, \phi_i(x_1)\, \phi_j(x_2)$$

$$\Sigma_\mathrm{noise}(x) = U(x)\, U(x)^\top + \delta\, I$$

where $\phi_i$ are Chebyshev basis functions and $\delta$ (`diag_term`) ensures
positive definiteness.  Here we evaluate this at the same 7×7 grid of reference
stimuli used in the paper.

In [ ]:
# 7×7 grid of reference stimuli in W-space (matches the paper's Figure 2C grid)
# The pkl stores Sigmas_thres_grid with W-dim-1 (x) varying along the column
# axis and W-dim-2 (y) along the row axis, matching the CSV ordering:
#   grid[i, j] = [pts[j], pts[i]]  →  x = pts[j], y = pts[i]
num_pts = model_pred.num_grid_pts1  # 7
pts = np.linspace(-0.7, 0.7, num_pts)
g_row, g_col = np.meshgrid(pts, pts, indexing="ij")
grid = np.stack([g_col, g_row], axis=-1)  # (7,7,2): grid[i,j] = [x=pts[j], y=pts[i]]

# Evaluate the W-field at every grid point
# compute_U: (degree,degree,2,3) × (7,7,2) → U shape (7,7,2,3)
# compute_Sigmas: U → Sigma_noise shape (7,7,2,2)
U = model.compute_U(W_est, jnp.array(grid))
Sigmas_noise = np.array(model.compute_Sigmas(U))

print(f"Grid shape:          {grid.shape}")
print(f"U shape:             {np.array(U).shape}")
print(f"Sigmas_noise shape:  {Sigmas_noise.shape}")
print(f"\nNoise Sigma at grid centre {(num_pts // 2, num_pts // 2)}:")
print(Sigmas_noise[num_pts // 2, num_pts // 2].round(6))

## 4. Threshold covariance matrices

Converting $\Sigma_\mathrm{noise}$ to $\Sigma_\mathrm{thres}$ (the boundary of
the 66.7 % correct discrimination contour) requires simulating the oddity task
for each reference location — about 16 000 trials per grid point × 49 grid points.
These have been pre-computed and stored in the pkl object as `Sigmas_thres_grid`.

In [ ]:
if not hasattr(model_pred, "Sigmas_thres_grid") or np.all(
    np.isnan(model_pred.Sigmas_thres_grid)
):
    raise RuntimeError(
        "Sigmas_thres_grid is missing or all-NaN in this pkl. "
        "Was convert_Sig_Threshold_oddity_batch run before pickling?"
    )

Sigmas_thres = np.array(model_pred.Sigmas_thres_grid)  # (7, 7, 2, 2)
centre = (num_pts // 2, num_pts // 2)

print(f"Sigmas_thres shape: {Sigmas_thres.shape}")
print(f"\nThreshold Sigma at grid centre {centre}:")
print(Sigmas_thres[centre].round(6))
print(f"\nNoise Sigma at grid centre {centre} (for comparison):")
print(Sigmas_noise[centre].round(6))
print(
    "\n(Threshold Sigma is larger — the oddity task requires a "
    "bigger stimulus difference than noise alone would predict.)"
)

## 5. Colour mapping

Each ellipse is coloured to approximately match the appearance of its reference
stimulus.  `color_thresholds.W2D_to_rgb()` converts a 2-D W-space coordinate to a
display RGB triplet via the monitor calibration matrices.

In [ ]:
color_thres = color_thresholds(2, str(DATA_DIR.resolve()), PLANE)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    color_thres.load_transformation_matrix(file_date="02242025")

print("Transformation matrices loaded.")
rgb_centre = color_thres.W2D_to_rgb(np.array([0.0, 0.0])).squeeze()
print(f"RGB at W-space origin: {rgb_centre.round(3)}")

## 6. Noise covariance ellipses (1 SD isoprobability contours)

The Wishart process model directly produces $\Sigma_\mathrm{noise}(x)$ at each grid point.
The 1 SD isoprobability contour — the ellipse $\{\mathbf{c} + \mathbf{L}\mathbf{u} : \|\mathbf{u}\| = 1\}$
where $\mathbf{L} = \mathrm{chol}(\Sigma_\mathrm{noise})$ — is plotted here.
These ellipses are expected to be roughly 2.6× smaller in radius than the threshold ellipses
and to have approximately the same shape.

In [ ]:
N_THETA = 200
theta = np.linspace(0, 2 * np.pi, N_THETA)
unit_circle = np.vstack([np.cos(theta), np.sin(theta)])  # (2, N_THETA)

noise_ellipses = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for idx in np.ndindex(num_pts, num_pts):
        center = grid[idx]  # (2,)
        Sigma = Sigmas_noise[idx]  # (2, 2)
        L = np.linalg.cholesky(Sigma)
        xy = center[:, None] + L @ unit_circle  # (2, N_THETA)
        rgb = color_thres.W2D_to_rgb(center).squeeze()
        noise_ellipses.append((xy, rgb))

print(f"Computed {len(noise_ellipses)} noise ellipses.")

fig, ax = plt.subplots(figsize=(5, 5), dpi=100)

for xy, rgb in noise_ellipses:
    ax.plot(xy[0], xy[1], color=rgb, lw=1.2)

ticks = np.linspace(-0.7, 0.7, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.set_xlabel("W dim 1", fontsize=9)
ax.set_ylabel("W dim 2", fontsize=9)
ax.set_title(
    f"Noise ellipses (1 SD) — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
    fontsize=10,
)
plt.tight_layout()
plt.show()
plt.close()

## 7. Reconstruct threshold ellipse outlines

Each 2×2 threshold covariance matrix $\boldsymbol{\Sigma}$ defines an ellipse:

$$\mathcal{E} = \{\mathbf{c} + \mathbf{L}\mathbf{u} : \|\mathbf{u}\| = 1\}$$

where $\mathbf{L}$ is the Cholesky factor and $\mathbf{c}$ is the reference location.

In [ ]:
# N_THETA and unit_circle are defined in section 6 above.
ellipses = []  # threshold ellipses
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for idx in np.ndindex(num_pts, num_pts):
        center = grid[idx]  # (2,)
        Sigma = Sigmas_thres[idx]  # (2, 2)
        L = np.linalg.cholesky(Sigma)
        xy = center[:, None] + L @ unit_circle  # (2, N_THETA)
        rgb = color_thres.W2D_to_rgb(center).squeeze()
        ellipses.append((xy, rgb))

print(f"Computed {len(ellipses)} threshold ellipses.")

## 8. Figure 2C — threshold and noise ellipses on the 7×7 grid

Threshold ellipses (solid) define the 66.7 % correct discrimination boundary.
Noise ellipses (dashed, same colour) show the 1 SD isoprobability contour of the
underlying noise distribution — expected to be roughly 2.6× smaller in radius.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5), dpi=100)

for xy, rgb in ellipses:
    ax.plot(xy[0], xy[1], color=rgb, lw=1.2)

for xy, rgb in noise_ellipses:
    ax.plot(xy[0], xy[1], color=rgb, lw=0.7, linestyle="--")

# Scaling check: upper-right noise ellipse × 2.6
ur_xy, _ = noise_ellipses[-1]  # idx (6,6) = upper-right grid point
ur_center = grid[6, 6]
ur_scaled = ur_center[:, None] + 2.6 * (ur_xy - ur_center[:, None])
ax.plot(ur_scaled[0], ur_scaled[1], color="black", lw=1.2, linestyle=":")

# Legend proxies
ax.plot([], [], color="gray", lw=1.2, label="threshold (66.7% correct)")
ax.plot([], [], color="gray", lw=0.7, linestyle="--", label="noise (1 SD)")
ax.plot([], [], color="black", lw=1.2, linestyle=":", label="noise × 2.6 (upper right)")
ax.legend(fontsize=7, loc="upper left")

ticks = np.linspace(-0.7, 0.7, 5)
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xlim(-1.0, 1.0)
ax.set_ylim(-1.0, 1.0)
ax.set_aspect("equal")
ax.set_xlabel("W dim 1", fontsize=9)
ax.set_ylabel("W dim 2", fontsize=9)
ax.set_title(
    f"Threshold & noise ellipses — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
    fontsize=10,
)
plt.tight_layout()
plt.show()
plt.close()

## 9. Monte Carlo threshold computation — lower-left point

As a sanity check we re-derive the threshold covariance for the lower-left
grid point ($x = y = -0.7$) by running the full oddity-task Monte Carlo pipeline
directly from the noise covariance field.  The result should be a close (though
noisy, given the reduced sample count) match to the pre-stored value in the pkl.

In [ ]:
from analysis.ellipses_tools import (  # noqa: E402
    ellParamsQ_to_covMat,
    fit_2d_isothreshold_contour,
)
from core import oddity_task  # noqa: E402

# ── Parameters ────────────────────────────────────────────────────────────────
N_THETA_TEST = 8  # directions around the ellipse
NGRID_TEST = 20  # vector lengths per direction
MC_SAMPLES = 500  # Monte Carlo samples per (direction, length) pair

bandwidth = model_pred.opt_params["bandwidth"]
# Use the module's simulate_oddity directly — the pkl's
# sim_func is a dill-serialized JAX vmap that crashes on
# current JAX versions.
sim_func = oddity_task.simulate_oddity
target_pC = model_pred.target_pC

w_ref = grid[0, 0]  # lower-left: [-0.7, -0.7]
vec_lengths = np.linspace(0.001, 0.25, NGRID_TEST)

thetas = np.linspace(0, 2 * np.pi, N_THETA_TEST, endpoint=False)
directions = np.column_stack([np.cos(thetas), np.sin(thetas)])  # (N_THETA_TEST, 2)

print(f"Lower-left point: {w_ref}")
print(f"  n_theta={N_THETA_TEST}, ngrid={NGRID_TEST}, mc_samples={MC_SAMPLES}")

# Pre-compute U at the reference (same for every direction)
w_ref_rep = np.tile(w_ref, (NGRID_TEST, 1))
Uref = model.compute_U(model_pred.W_est, jnp.array(w_ref_rep))  # (NGRID, 2, 3)

recover_vec_lengths = np.zeros(N_THETA_TEST)

for i, d in enumerate(directions):
    w_comps = w_ref[None, :] + vec_lengths[:, None] * d[None, :]
    U1 = model.compute_U(model_pred.W_est, jnp.array(w_comps))  # (NGRID, 2, 3)

    keys_i = jax.random.split(jax.random.fold_in(jax.random.PRNGKey(0), i), NGRID_TEST)
    pC = np.array(
        oddity_task.oddity_prediction(
            (jnp.array(w_ref_rep), jnp.array(w_comps), Uref, U1),
            keys_i,
            MC_SAMPLES,
            bandwidth,
            model.diag_term,
            sim_func,
        )
    )  # (NGRID_TEST,)

    recover_vec_lengths[i] = vec_lengths[np.argmin(np.abs(pC - target_pC))]
    print(
        f"  dir {i}: pC [{pC.min():.3f}, {pC.max():.3f}]  "
        f"threshold={recover_vec_lengths[i]:.4f}"
    )

# Contour points: (2, N_THETA_TEST)
w_comp_est = w_ref[:, None] + directions.T * recover_vec_lengths[None, :]

_, _, params_ell, _ = fit_2d_isothreshold_contour(
    w_ref, w_comp_est, nTheta=200, flag_force_centered_ref=True
)
Sigma_thres_mc = np.array(ellParamsQ_to_covMat(*params_ell[2:]))

print("MC threshold Sigma:")
print(Sigma_thres_mc.round(6))
print("Stored threshold Sigma:")
print(Sigmas_thres[0, 0].round(6))

# ── Comparison plot ────────────────────────────────────────────────────────────
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    rgb_ll = color_thres.W2D_to_rgb(w_ref).squeeze()

fig, ax = plt.subplots(figsize=(3.5, 3.5), dpi=100)
for label, Sigma, color, ls, lw in [
    ("stored threshold", Sigmas_thres[0, 0], rgb_ll, "-", 1.6),
    ("MC threshold", Sigma_thres_mc, "black", "--", 1.2),
]:
    L = np.linalg.cholesky(Sigma)
    xy = w_ref[:, None] + L @ unit_circle
    ax.plot(xy[0], xy[1], color=color, lw=lw, linestyle=ls, label=label)

ax.plot(*w_comp_est, "k+", ms=5, label="MC threshold pts")
ax.set_aspect("equal")
ax.legend(fontsize=8)
ax.set_title("Lower-left MC threshold check", fontsize=9)
plt.tight_layout()
plt.show()
plt.close()

## 10. Bootstrap confidence regions

### 10a. Load bootstrap fit objects

Each bootstrap pkl contains a `wishart_model_pred` object fitted to a bootstrap
resample of the data.  We extract the pre-stored `Sigmas_thres_grid` from each
— no re-computation needed.

In [ ]:
btst_pkls = sorted(glob_mod.glob(str(PKL_DIR / "*btst*.pkl")))
N_BTST = len(btst_pkls)
print(f"Found {N_BTST} bootstrap pkl files", end="")

if N_BTST == 0:
    print(
        "\n\nNo bootstrap pkl files found.  Re-run download_data.py with --fits "
        "to fetch them, then restart and re-run this cell.\n"
        "The CI cells below will be skipped."
    )
    Sigmas_btst = None
else:
    print(" — loading ...")
    btst_list = []
    for k, p in enumerate(btst_pkls):
        with open(p, "rb") as fh:
            vd = _RobustUnpickler(fh).load()
        btst_list.append(np.array(vd["model_pred_Wishart"].Sigmas_thres_grid))
        if (k + 1) % 20 == 0:
            print(f"  loaded {k + 1}/{N_BTST}")
    # shape: (N_BTST, 7, 7, 2, 2) → rearrange to (7, 7, N_BTST, 2, 2)
    Sigmas_btst = np.stack(btst_list, axis=2)
    print(f"Bootstrap Sigmas shape: {Sigmas_btst.shape}")

### 10b. Rank bootstraps by Normalized Bures Similarity and retain top 95 %

For each bootstrap we sum the NBS score between the bootstrap $\Sigma_\mathrm{thres}$
and the original across all 49 grid points.  The top 95 % (114 of 120) are retained,
following Hong et al. (2025).

In [ ]:
if Sigmas_btst is not None:

    def nbs(M1: npt.NDArray[np.float64], M2: npt.NDArray[np.float64]) -> float:
        """Normalized Bures Similarity between two 2×2 PD matrices."""
        inner = scipy_sqrtm(scipy_sqrtm(M1) @ M2 @ scipy_sqrtm(M1))
        return float(np.real(np.trace(inner)) / np.sqrt(np.trace(M1) * np.trace(M2)))

    CI_FRACTION = 0.95
    N_KEEP = max(1, int(N_BTST * CI_FRACTION))  # 114 of 120
    print(f"Retaining top {N_KEEP} of {N_BTST} bootstrap datasets")

    nbs_sum = np.zeros(N_BTST)
    for b in range(N_BTST):
        for idx in np.ndindex(num_pts, num_pts):
            nbs_sum[b] += nbs(Sigmas_thres[idx], Sigmas_btst[idx][b])

    idx_keep = np.argsort(nbs_sum)[::-1][:N_KEEP]
    Sigmas_btst_kept = Sigmas_btst[:, :, idx_keep, :, :]  # (7, 7, 114, 2, 2)
    print(
        f"NBS sum range (kept): "
        f"{nbs_sum[idx_keep].min():.4f} – {nbs_sum[idx_keep].max():.4f}"
    )

### 10c. Compute radial CI envelopes

For each direction $\theta$, the radial distance from ellipse centre to boundary is
$r(\theta) = 1/\sqrt{v^\top \Sigma^{-1} v}$ where $v = [\cos\theta, \sin\theta]$.
Taking the min/max across retained bootstraps gives the inner/outer CI envelope.

In [ ]:
if Sigmas_btst is not None:
    N_THETA_CI = 360
    theta_ci = np.linspace(0, 2 * np.pi, N_THETA_CI, endpoint=False)
    dirs = np.column_stack([np.cos(theta_ci), np.sin(theta_ci)])  # (360, 2)

    r_btst = np.full((num_pts, num_pts, N_KEEP, N_THETA_CI), np.nan)
    for idx in np.ndindex(num_pts, num_pts):
        for k in range(N_KEEP):
            S_inv = np.linalg.inv(Sigmas_btst_kept[idx][k])
            quad = np.einsum("td,de,te->t", dirs, S_inv, dirs)
            r_btst[idx][k] = 1.0 / np.sqrt(quad)

    r_outer = r_btst.max(axis=2)  # (7, 7, 360)
    r_inner = r_btst.min(axis=2)
    print(f"r_outer shape: {r_outer.shape}")

### 10d. Figure 2C with 95 % bootstrap confidence regions

In [ ]:
if Sigmas_btst is None:
    print("Bootstrap pkls not loaded — skipping CI plot.")
else:
    fig, ax = plt.subplots(figsize=(5, 5), dpi=100)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for idx in np.ndindex(num_pts, num_pts):
            center = grid[idx]
            rgb = color_thres.W2D_to_rgb(center).squeeze()

            outer_xy = center[:, None] + r_outer[idx] * dirs.T
            inner_xy = center[:, None] + r_inner[idx] * dirs.T
            band_x = np.concatenate([outer_xy[0], inner_xy[0, ::-1], [outer_xy[0, 0]]])
            band_y = np.concatenate([outer_xy[1], inner_xy[1, ::-1], [outer_xy[1, 0]]])
            ax.fill(band_x, band_y, color=rgb, alpha=0.4, linewidth=0)

            L = np.linalg.cholesky(Sigmas_thres[idx])
            theta_plot = np.linspace(0, 2 * np.pi, 200)
            uc = np.vstack([np.cos(theta_plot), np.sin(theta_plot)])
            xy = center[:, None] + L @ uc
            ax.plot(xy[0], xy[1], color="black", lw=0.8)

        for xy, rgb in noise_ellipses:
            ax.plot(xy[0], xy[1], color=rgb, lw=0.7, linestyle="--")

    # Legend proxies
    ax.plot([], [], color="black", lw=0.8, label="threshold (66.7% correct)")
    ax.plot([], [], color="gray", lw=0.7, linestyle="--", label="noise (1 SD)")
    ax.legend(fontsize=7, loc="upper left")

    ticks = np.linspace(-0.7, 0.7, 5)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xlim(-1.0, 1.0)
    ax.set_ylim(-1.0, 1.0)
    ax.set_aspect("equal")
    ax.set_xlabel("W dim 1", fontsize=9)
    ax.set_ylabel("W dim 2", fontsize=9)
    ax.set_title(
        f"Threshold + noise + 95% CI — sub{SUB_N} ({SUB_INITIALS[SUB_N]}) — {PLANE}",
        fontsize=9,
    )
    plt.tight_layout()
    plt.show()
    plt.close()